# Comprehensive Exploratory Data Analysis (EDA): Nigerian Electricity Access
**Author:** Raji Quadri (Adetunji)  
**Role:** Data Analyst & AI Engineer  
**Dataset Reference:** 2021 MIS Nigerian Electricity Distribution Dataset  

## 📌 Project Background & Objectives
This notebook conducts a granular exploratory data analysis (EDA) of household and population-level electricity access across Nigeria's 36 states and the Federal Capital Territory (FCT). The analysis identifies regional disparities, infrastructure gaps, and provides actionable insights for policy recommendations.

In [ ]:
# ===== CONSOLIDATED ENVIRONMENT SETUP =====
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Verify versions
print("=== ENVIRONMENT STACK ===")
print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Seaborn: {sns.__version__}")

# Configure visualization
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.figsize'] = (14, 8)
sns.set_theme(style="whitegrid", palette="husl")

In [ ]:
# ===== DATA LOADING WITH VALIDATION =====
def load_electricity_data(folder="New_Project_Dataset", filename="Nigeria_Electricity_Data.xlsx"):
    """Load and validate Nigerian electricity dataset.
    
    Args:
        folder (str): Directory containing the dataset
        filename (str): Name of the Excel file
        
    Returns:
        pd.DataFrame: Loaded electricity data
        
    Raises:
        FileNotFoundError: If dataset not found in expected locations
    """
    paths = [os.path.join(folder, filename), filename]
    
    for path in paths:
        if os.path.exists(path):
            print(f"✓ Loading: {path}")
            df = pd.read_excel(path)
            print(f"✓ Shape: {df.shape}")
            return df
    
    raise FileNotFoundError(f"Dataset not found. Checked: {paths}")

df = load_electricity_data()

# Data validation
print(f"\n✓ Missing values: {df.isnull().sum().sum()}")
print(f"✓ Duplicates: {df.duplicated().sum()}")

In [ ]:
# ===== DATA QUALITY CHECK =====
# Flag values >100% (data anomalies)
invalid_household = (df['Households with electricity'] > 100).sum()
invalid_population = (df['Population with electricity'] > 100).sum()

print(f"Values >100%: Households={invalid_household}, Population={invalid_population}")
print(f"\n=== STRUCTURAL SCHEMA ===")
df.info()
print(f"\n=== FIRST 10 RECORDS ===")
df.head(10)

In [ ]:
# ===== STATISTICAL SUMMARY =====
print("=== DESCRIPTIVE STATISTICS ===")
summary = df[['Households with electricity', 'Population with electricity']].describe().T
summary['range'] = summary['max'] - summary['min']
summary['cv'] = (summary['std'] / summary['mean']) * 100  # Coefficient of variation
print(summary)

# Key insights
print(f"\n📊 KEY FINDINGS:")
print(f"  • Mean household access: {summary.loc['Households with electricity', 'mean']:.2f}%")
print(f"  • Std deviation: {summary.loc['Households with electricity', 'std']:.2f}%")
print(f"  • Range (Max-Min): {summary.loc['Households with electricity', 'range']:.2f}%")
print(f"  • Coefficient of variation: {summary.loc['Households with electricity', 'cv']:.2f}%")
print(f"\n  ➜ High inequality distribution: Mean={summary.loc['Households with electricity', 'mean']:.2f}%, Std={summary.loc['Households with electricity', 'std']:.2f}%")
print(f"  ➜ Extreme regional disparities: {summary.loc['Households with electricity', 'range']:.2f}% range between highest and lowest")

In [ ]:
# ===== ACCESS TIER CLASSIFICATION =====
def classify_access(pct):
    """Classify states into infrastructure tiers based on access percentage.
    
    Args:
        pct (float): Electricity access percentage
        
    Returns:
        str: Access tier classification
    """
    if pct >= 75.0:
        return 'High Access (≥75%)'
    elif pct >= 40.0:
        return 'Medium Access (40-75%)'
    else:
        return 'Critical Need (<40%)'

df['Access_Tier'] = df['Households with electricity'].apply(classify_access)

# Sort by access level
df_sorted = df.sort_values('Households with electricity', ascending=False).reset_index(drop=True)

print("=== INFRASTRUCTURE TIER DISTRIBUTION ===")
tier_counts = df['Access_Tier'].value_counts()
print(tier_counts)
print(f"\nFull Ranking:\n{df_sorted[['State', 'Households with electricity', 'Access_Tier']].to_string()}")

In [ ]:
# ===== CORRELATION ANALYSIS =====
correlation = df['Households with electricity'].corr(df['Population with electricity'])
print(f"Correlation (Households vs Population): {correlation:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['Households with electricity'], df['Population with electricity'], 
          s=100, alpha=0.6, color='steelblue', edgecolors='navy', linewidth=1.5)
ax.set_xlabel('Households with Electricity (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Population with Electricity (%)', fontsize=12, fontweight='bold')
ax.set_title(f'Electricity Access: Households vs Population (r={correlation:.3f})', 
            fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Add state labels for significant deviations
for idx, row in df.iterrows():
    if abs(row['Households with electricity'] - row['Population with electricity']) > 5:
        ax.annotate(row['State'], 
                   (row['Households with electricity'], row['Population with electricity']), 
                   fontsize=8, alpha=0.7, xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
# ===== IMPROVED HORIZONTAL BAR CHART =====
fig, ax = plt.subplots(figsize=(12, 14))

# Create color mapping based on access tier
colors = df_sorted['Access_Tier'].map({
    'High Access (≥75%)': '#2ecc71',
    'Medium Access (40-75%)': '#f39c12',
    'Critical Need (<40%)': '#e74c3c'
})

# Create bars
bars = ax.barh(df_sorted['State'], df_sorted['Households with electricity'], 
               color=colors, edgecolor='black', linewidth=0.8)

# Formatting
ax.set_xlabel('Household Electricity Access (%)', fontsize=12, fontweight='bold')
ax.set_title('Nigerian States: Household Electricity Access Rate', 
            fontsize=14, fontweight='bold', pad=20)
ax.set_xlim(0, 100)

# Add value labels on bars
for i, (idx, row) in enumerate(df_sorted.iterrows()):
    ax.text(row['Households with electricity'] + 1, i, f"{row['Households with electricity']:.1f}%", 
           va='center', fontsize=9, fontweight='bold')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', edgecolor='black', label='High Access (≥75%)'),
    Patch(facecolor='#f39c12', edgecolor='black', label='Medium Access (40-75%)'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='Critical Need (<40%)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ===== SUMMARY EXPORT =====
summary_export = df_sorted[['State', 'Households with electricity', 'Population with electricity', 'Access_Tier']].copy()
summary_export.to_csv('Nigeria_Electricity_Summary.csv', index=False)
print("✓ Summary exported to: Nigeria_Electricity_Summary.csv")

print("\n=== ANALYSIS COMPLETE ===")
print(f"Total states analyzed: {len(df)}")
print(f"States with high access: {(df['Access_Tier'] == 'High Access (≥75%)').sum()}")
print(f"States with medium access: {(df['Access_Tier'] == 'Medium Access (40-75%)').sum()}")
print(f"States requiring critical investment: {(df['Access_Tier'] == 'Critical Need (<40%)').sum()}")

## 📈 Summary of Key Insights

### Regional Disparities
- **High Inequality Distribution:** The average household connection rate is approximately **49.33%**, yet with a standard deviation of **25.60%**, indicating significant regional variation.
- **Extreme Discrepancies:** There is a massive range of **94.9%** between the most and least electrified states (Max: 97.6% vs. Min: 2.7%).

### Infrastructure Tiers
- **High Access States:** Kwara, Lagos, and other southern states exceed 75% coverage
- **Medium Access States:** Central and some northern states with 40-75% coverage
- **Critical Need States:** Primarily northern states (Borno, Bauchi, Plateau) requiring urgent investment

### Next Steps
1. Implement targeted electrification programs for critical need states
2. Analyze root causes of regional disparities (geography, infrastructure, population density)
3. Develop predictive models for infrastructure expansion planning